In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim #用来更新参数
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# 这个项目有一点点的问题，数据集没有划分为训练集和测试集，但是算了没关系，后面可以学习这个。

In [47]:
#准备数据：用numpy随机构造一些线性数据

np.random.seed(42)#设置随机数种子，使得每次运行代码时生成的随机数相同

x_np = np.random.rand(1000,3).astype(np.float32) #生成1000行3列的随机数，并转换为float32类型
true_w = [2.0, -3.4, 4.5] #定义真实的权重
true_b = 4.2 #定义真实的偏置
y_np = (np.dot(x_np,true_w) + true_b).reshape(1000,1) + np.random.normal(0,0.1,(1000,1)).astype(np.float32) #根据线性关系生成标签，并添加一些高斯噪声

print(x_np.shape, y_np.shape) #打印输入数据和标签的形状

(1000, 3) (1000, 1)


In [48]:
#数据标准化，防止数据间绝对值差距太大

mean = x_np.mean(axis=0) #计算每列的均值
std = x_np.std(axis=0) #计算每列的标准差
x_np = (x_np - mean) / std #对每列进行标准化

In [49]:
#转化为tensor
x = torch.from_numpy(x_np).float() #将numpy数组转换为torch张量
y = torch.from_numpy(y_np).float() #将numpy数组转换为torch张量
print(x.dtype, y.dtype)

torch.float32 torch.float32


In [50]:
#用dataloader加载数据（防止一次性数据量太大
# dataset: 组合特征和标签；batch_size: 批大小；shuffle: 训练集必须打乱

dataset = TensorDataset(x,y) #创建一个TensorDataset对象，将特征和标签组合在一起
#它把你的特征 x 和标签 y 一对一地缝合在一起。不需要手动维护防止特征和标签的index取错。把(x,y)变成一个整体。
#batchsize设置为2的幂指数（如32、64、128等）通常是因为计算机的内存和处理器架构更适合处理这样的数据块大小，这样可以提高训练效率。
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
# batch_size=32: 每次训练迭代中，模型将使用32个样本来计算梯度和更新参数。这有助于平衡训练效率和模型性能。
# shuffle=True: 在每个训练周期开始时，数据将被随机打乱。这有助于防止模型在训练过程中学习到数据的顺序，从而提高模型的泛化能力。
#dataset和dataloader的关系：dataset是一个数据集对象，包含了所有的数据样本和标签；dataloader是一个迭代器，用于从dataset中按批次加载数据，并提供一些额外的功能，如打乱数据、并行加载等。



In [51]:
#定义模型
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__() #调用父类的构造函数
        self.linear = nn.Linear(3,1) #定义一个线性层，输入特征数为3，输出特征数为1
    def forward(self,x):
        out = self.linear(x) #前向传播，计算线性层的输出
        return out
    
model = LinearRegressionModel() #创建模型实例

这段代码定义了一个**线性回归机器**。我们从程序逻辑的角度，逐行、逐词地拆解它是如何“诞生”并“准备工作”的。

### 1. 第一部分：图纸定义 (Defining the Blueprint)

```python
class LinearRegressionModel(nn.Module):
```
*   **`class`**: 关键字，告诉 Python “我要开始画图纸了”。
*   **`LinearRegressionModel`**: 这是你给这个机器起的名字。
*   **`(nn.Module)`**: **继承**。表示这个机器是基于 PyTorch 官方的标准外壳改造的，它会自动获得计算梯度等“超能力”。

---

### 2. 第二部分：零件准备 (Initialization - `__init__`)

```python
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(3, 1)
```
**程序逻辑：** 当你创建模型实体时，这段代码会运行一次，把零件装好。

*   **`def __init__(self)`**: 构造函数。`self` 代表模型未来的那个“实体”，就像在说“我出厂时要带的东西”。
*   **`super(...).__init__()`**: 
    *   **`super()`**: 找到父类 `nn.Module`。
    *   **作用**：激活官方底座。如果不运行这句，你的机器连电源（参数追踪功能）都没接通。
*   **`self.linear = nn.Linear(3, 1)`**:
    *   **`self.linear`**: 给这个零件起个名字叫 `linear`，并存在“我”自己身上。
    *   **`nn.Linear(3, 1)`**: 这是 PyTorch 提供的线性零件。
    *   **参数 `3` (Input)**: 告诉它输入数据有 3 个特征（对应你数据里的 3 列）。
    *   **参数 `1` (Output)**: 预测出 1 个结果（对应你要预测的 y 值）。
    *   **隐藏动作**：这一行运行完，模型内部会自动生成 3 个权重 $w$ 和 1 个偏置 $b$。

---

### 3. 第三部分：运行逻辑 (The Work Flow - `forward`)

```python
    def forward(self, x):
        out = self.linear(x)
        return out
```
**程序逻辑：** 定义数据进站后，怎么走流程。

*   **`forward`**: 这是一个**特殊的方法名**。PyTorch 规定，当你把数据丢给模型（`model(x)`）时，会自动跑这个函数。
*   **`(self, x)`**: `x` 是输入进来的原材料（数据张量）。
*   **`out = self.linear(x)`**: 让数据 `x` 穿过刚才准备好的那个 `linear` 零件。内部执行的就是数学运算：$Y = X \cdot W + b$。
*   **`return out`**: 把计算结果（成品）吐出来。

---

### 4. 第四部分：实例化 (Creation)

```python
model = LinearRegressionModel()
```
*   这一行真正按照图纸**造出了**一个机器，名叫 `model`。此时，`__init__` 里的代码被执行，零件全部各就各位。

---

### 输入与输出 (Input & Output)

| 环节 | 数据形状 (Shape) | 含义 |
| :--- | :--- | :--- |
| **输入 (Input `x`)** | `(Batch_Size, 3)` | 比如一次喂入 32 条数据，每条数据 3 个特征。 |
| **计算流程** | `32x3` 矩阵 × `3x1` 权重 + `1` 偏置 | 内部矩阵运算。 |
| **输出 (Output `out`)** | `(Batch_Size, 1)` | 得到 32 个预测值，每个值对应一条数据。 |

**简单总结运行逻辑：**
1.  **定义阶段**：确定需要一个 3 进 1 出的线性零件。
2.  **创建阶段**：实例化模型，自动初始化了 $w$ 和 $b$。
3.  **计算阶段（未来）**：数据 `x` 进来 $\to$ `forward` 启动 $\to$ 计算 $x \cdot w + b$ $\to$ 返回预测值。

In [52]:
#损失函数

criterion = nn.MSELoss() #均方误差损失函数，适用于回归问题

# 优化器：随机梯度下降 SGD
# model.parameters() 会把模型中所有待训练的 w 和 b 传给优化器
# lr: 学习率 (Learning Rate)
optimizer = optim.SGD(model.parameters(), lr=0.001) #随机梯度下降优化器，学习率为0.01

In [53]:
#训练流程
epochs = 100#训练轮数

for epoch in range(epochs):
    for batch_idx, (batch_x, batch_y) in enumerate(train_loader): #遍历每个批次的数据
    #for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        #前向传播
        prediction = model(batch_x) #计算模型的输出
        #计算损失
        loss = criterion(prediction, batch_y)

        #梯度清零
        optimizer.zero_grad() #清除之前的梯度信息，防止累积

        #反向传播
        loss.backward() #计算当前损失相对于模型参数的梯度

        #更新参数
        optimizer.step() #根据计算的梯度更新模型参数
    if (epoch+1) % 10 == 0: #每10轮打印一次损失
        print(f'epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f} ')

epoch [10/100], Loss: 11.9026 
epoch [20/100], Loss: 3.6149 
epoch [30/100], Loss: 0.6156 
epoch [40/100], Loss: 0.2112 
epoch [50/100], Loss: 0.0692 
epoch [60/100], Loss: 0.0303 
epoch [70/100], Loss: 0.0141 
epoch [80/100], Loss: 0.0224 
epoch [90/100], Loss: 0.0152 
epoch [100/100], Loss: 0.0151 


这是一个非常核心的问题。答案在于：**PyTorch 模型内部执行的是“矩阵运算”**，而矩阵运算天然支持同时处理多条数据。

我们可以从两个层面来理解：

### 1. 形状的匹配（Shape Alignment）
在你的代码中，模型内部的 `nn.Linear(3, 1)` 实际上持有一个形状为 `(1, 3)` 的权重矩阵 $W$。

*   **单条数据**：输入 `x` 是 `(1, 3)`。计算是 `(1, 3) × (3, 1)`，结果是 `(1, 1)`。
*   **32 条数据 (`batch_x`)**：输入 `batch_x` 是一个形状为 **`(32, 3)`** 的矩阵。
    *   **计算过程**：PyTorch 执行矩阵乘法 `(32, 3) × (3, 1)`。
    *   **数学结果**：根据矩阵乘法规则，结果是一个 **`(32, 1)`** 的矩阵。
    *   这个结果矩阵中的第 1 行就是第 1 条数据的预测，第 2 行就是第 2 条。

这种一口气算出 32 个结果的能力，在计算机科学中叫作**“向量化计算”**。

### 2. 为什么这样做更高效？
如果 `model` 每次只能处理 1 条数据，你要训练 1000 条数据就得像剥洋葱一样写 1000 个 `for` 循环。而 CPU/GPU 处理器非常擅长这种**批量**操作：

*   **硬件层面**：GPU 内部有成千上万个微小的核心。当你丢进一个 `(32, 3)` 的矩阵时，GPU 会把这 32 条数据的计算任务同时分配给不同的核心。
*   **速度对比**：同时算 32 条数据的时间，通常和只算 1 条数据的时间**几乎一样快**。

### 3. 谁在背后调度？
当你写 `prediction = model(batch_x)` 时：
1.  **`__call__` 触发**：PyTorch 检测到你传入了一个 Tensor。
2.  **`forward` 被调用**：它把这个 `(32, 3)` 的大矩阵传给了你写的 `self.linear(x)`。
3.  **底层计算库（ATen）**：它会识别出这是一个批次数据，并调用优化的矩阵乘法函数（BLAS 或 cuBLAS）来完成运算。

---

### 总结
`model(batch_x)` 之所以能“塞进” 32 条数据，是因为：
1.  **数学上**：采用了矩阵乘法，输入多少行，输出就有多少行。
2.  **代码上**：PyTorch 的层（Layers）都被设计成**“批次优先（Batch-first）”**。它默认输入的第一个维度就是批次大小。

你可以试着打印一下看看：
```python
print(batch_x.shape)    # 输出 torch.Size([32, 3])
prediction = model(batch_x)
print(prediction.shape) # 输出 torch.Size([32, 1])


这是一个关于 Python 语法和 PyTorch 设计的非常经典的问题。我们分三点来拆解：

### 1. 为什么可以有两个变量？（Python 的“解包”语法）
Python 支持一种叫 **“序列解包 (Unpacking)”** 的特性。当一个列表里的每个元素本身也是一个包含多个项的元组（Tuple）时，你可以在 `for` 循环中直接用多个变量接住它们。

*   **举个简单例子：**
    ```python
    data = [(1, "苹果"), (2, "香蕉")]
    for id, name in data:
        print(id, name) # 第一次循环输出 1 苹果，第二次输出 2 香蕉
    ```

### 2. 这两个变量的名字可以换吗？
**完全可以换！** 它们不是内部设计好的死名字，而是由你定义的**临时占位符**。

*   你写 `for batch_x, batch_y in train_loader` 是为了可读性（x 代表特征，y 代表标签）。
*   你完全可以写成 `for data, label in train_loader` 或者 `for a, b in train_loader`。
*   **唯一的要求**：变量的数量必须和 `train_loader` 每次吐出来的东西的数量一致。

### 3. 这两个变量的结构是怎么确定的？
这两个变量的结构**确实是和你的 `Dataset` 结构一模一样的**。

回顾你前面的代码：
```python
dataset = TensorDataset(x, y) # 你在这里塞进了两个东西：x 和 y
train_loader = DataLoader(dataset, ...)
```

*   **对应关系**：你在创建 `TensorDataset` 时按顺序传了 `(x, y)`，那么 `DataLoader` 在遍历时，每一批次返回的就是一个包含两个 Tensor 的元组 `(这一批的x, 这一批的y)`。
*   **如果你改了 Dataset**：假设你定义数据集时写的是 `TensorDataset(x, y, z)`，那么你的循环就必须写成 `for batch_x, batch_y, batch_z in train_loader`。

### 4. 为什么你的代码里还有一个 `batch_idx`？
现在的代码增加了 `enumerate`：
```python
for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
```
*   **`enumerate`** 的作用：它在 `train_loader` 吐出的 `(batch_x, batch_y)` 外面又套了一层，额外给了你一个从 0 开始递增的数字（即当前的批次索引）。
*   **结构变化**：原本返回是 `(x, y)`，用了 `enumerate` 变成了 `(index, (x, y))`。所以你需要对应的三个变量来接住它们。

**总结：** 
变量名是你定的，但**变量的数量和顺序**必须严格遵循 `DataLoader`（也就是你最初定义的 `Dataset`）的产出结构。这就好比流水线上每送来一个包裹，里面固定装了一件上衣和一条裤子，你可以叫它们 `a, b`，也可以叫 `top, pants`，但你必须用两个手去接。

In [ ]:
# 查看训练后的参数
print("\n训练后的权重 (w):", model.linear.weight.data)
print("训练后的偏置 (b):", model.linear.bias.data)
# 验证：将标准化后的权重还原回原始尺度
recovered_w = model.linear.weight.data.numpy() / std
recovered_b = model.linear.bias.data.item() - np.sum((model.linear.weight.data.numpy() * mean) / std)

print("训练出的权重 (w'): ", model.linear.weight.data.numpy())
print("还原后的权重 (理论应接近 [2.0, -3.4, 4.5]):\n", recovered_w)
print("\n还原后的偏置 (理论应接近 4.2):", recovered_b)

# 模型测试模式
#模型测试模式
model.eval() 
with torch.no_grad(): # 关闭梯度计算
    # 1. 随机抽取 20 个索引
    indices = np.random.choice(len(x), 20, replace=False)
    test_input = x[indices]
    test_target = y[indices]

    # 2. 进行预测
    predicted = model(test_input)

    # 3. 计算回归准确率 (假设预测值与实际值误差在 10% 以内认为准确)
    # 对于回归任务，通常查看平均绝对误差 (MAE) 或设定一个阈值
    diff = torch.abs(predicted - test_target)
    threshold = 0.1 * torch.abs(test_target) # 误差阈值设为实际值的 10%
    correct = (diff < threshold).float().sum()
    accuracy = (correct / 20) * 100

    # 4. 打印对比结果
    print(f"{'索引':<6} | {'预测值':<10} | {'实际值':<10} | {'误差':<10}")
    print("-" * 45)
    for i in range(20):
        print(f"{indices[i]:<8} | {predicted[i].item():<10.4f} | {test_target[i].item():<10.4f} | {diff[i].item():<10.4f}")
    
    print(f"\n基于误差阈值 (10%) 的预测准确率: {accuracy.item():.2f}%")

# 保存模型 (仅保存参数)
torch.save(model.state_dict(), 'linear_model.pth')

# 加载模型逻辑
# new_model = LinearRegressionModel(input_dim=3)
# new_model.load_state_dict(torch.load('linear_model.pth'))


训练后的权重 (w): tensor([[ 0.5742, -1.0043,  1.3079]])
训练后的偏置 (b): tensor([5.7552])
训练出的权重 (w'):  [[ 0.5742097 -1.0042664  1.3078778]]
还原后的权重 (理论应接近 [2.0, -3.4, 4.5]):
 [[ 2.0028734 -3.3913057  4.4859767]]

还原后的偏置 (理论应接近 4.2): 4.192161560058594
索引     | 预测值        | 实际值        | 误差        
---------------------------------------------
59       | 2.4069     | 2.2926     | 0.1143    
82       | 6.0943     | 6.1037     | 0.0094    
568      | 7.5888     | 7.6133     | 0.0244    
622      | 4.3106     | 4.1819     | 0.1287    
388      | 5.4731     | 5.6405     | 0.1673    
554      | 8.7990     | 8.6621     | 0.1369    
103      | 3.5979     | 3.5306     | 0.0672    
775      | 3.3271     | 3.3991     | 0.0719    
400      | 5.7267     | 5.9080     | 0.1813    
831      | 5.4410     | 5.5024     | 0.0614    
49       | 6.2744     | 6.2912     | 0.0168    
687      | 4.5440     | 4.5669     | 0.0228    
750      | 4.2973     | 4.2357     | 0.0616    
613      | 5.3182     | 5.2226     | 0.0956